# Feature Engineering

In this notebook, we create new features from the prepared data that will help the forecasting model learn historical sales patterns and improve prediction accuracy.


In [8]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)

In [9]:
sales = pd.read_csv(r"C:\Users\AHMAD\Desktop\data final project colledge\Depi M5\data\raw\sales_train_evaluation.csv")
calendar = pd.read_csv(r"C:\Users\AHMAD\Desktop\data final project colledge\Depi M5\data\raw\calendar.csv")
prices = pd.read_csv(r"C:\Users\AHMAD\Desktop\data final project colledge\Depi M5\data\raw\sell_prices.csv")

In [10]:
print("Sales:", sales.shape)
print("Calendar:", calendar.shape)
print("Prices:", prices.shape)

Sales: (30490, 1947)
Calendar: (1969, 14)
Prices: (6841121, 4)


## Prepare Time Series Data

Instead of creating a single large modeling dataset, we prepare a representative time series for forecasting. This approach reduces memory usage while preserving the complete forecasting workflow, allowing feature engineering and model development to be performed efficiently.

## Select Representative Store

Based on the exploratory data analysis, the CA_3 store achieved the highest cumulative sales among all stores. Therefore, it is selected as the representative store for building the forecasting model.

In [11]:
ca3_sales = sales[
    sales["store_id"] == "CA_3"
].copy()

In [12]:
ca3_sales.shape

(3049, 1947)

In [13]:
ca3_sales[
    ["id","item_id","dept_id","cat_id"]
].head()

,id,item_id,dept_id,cat_id
6098,HOBBIES_1_001_CA_3_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES
6099,HOBBIES_1_002_CA_3_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES
6100,HOBBIES_1_003_CA_3_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES
6101,HOBBIES_1_004_CA_3_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES
6102,HOBBIES_1_005_CA_3_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES


## Select Representative Product

To build a representative forecasting model, we select the product with the highest cumulative sales within the CA_3 store. This ensures that the selected time series contains sufficient historical demand and reflects a meaningful sales pattern.

In [14]:
daily_cols = ca3_sales.columns[6:]

product_sales = (
    ca3_sales.assign(total_sales=ca3_sales[daily_cols].sum(axis=1))
    .sort_values("total_sales", ascending=False)
)

product_sales[
    ["id", "item_id", "dept_id", "cat_id", "total_sales"]
].head(10)

,id,item_id,dept_id,cat_id,total_sales
8412,FOODS_3_090_CA_3_evaluation,FOODS_3_090,FOODS_3,FOODS,253859
8908,FOODS_3_586_CA_3_evaluation,FOODS_3_586,FOODS_3,FOODS,136269
8442,FOODS_3_120_CA_3_evaluation,FOODS_3_120,FOODS_3,FOODS,90412
8574,FOODS_3_252_CA_3_evaluation,FOODS_3_252,FOODS_3,FOODS,82861
8863,FOODS_3_541_CA_3_evaluation,FOODS_3_541,FOODS_3,FOODS,80495
8957,FOODS_3_635_CA_3_evaluation,FOODS_3_635,FOODS_3,FOODS,80252
9127,FOODS_3_808_CA_3_evaluation,FOODS_3_808,FOODS_3,FOODS,72597
8909,FOODS_3_587_CA_3_evaluation,FOODS_3_587,FOODS_3,FOODS,71200
8877,FOODS_3_555_CA_3_evaluation,FOODS_3_555,FOODS_3,FOODS,59999
9003,FOODS_3_681_CA_3_evaluation,FOODS_3_681,FOODS_3,FOODS,56790


## Create Time Series

The selected product is extracted and transformed from the original wide format into a chronological time series, where each row represents one day of sales. This structure is required for time series feature engineering and forecasting models.

In [15]:
top_item = product_sales.iloc[0]["item_id"]

top_product = ca3_sales[
    ca3_sales["item_id"] == top_item
].copy()

top_product[
    ["id", "item_id", "dept_id", "cat_id"]
]

,id,item_id,dept_id,cat_id
8412,FOODS_3_090_CA_3_evaluation,FOODS_3_090,FOODS_3,FOODS


In [16]:
daily_cols = top_product.columns[6:]

time_series = (
    top_product[daily_cols]
    .T
    .reset_index()
)

time_series.columns = ["d", "sales"]

time_series.head()

,d,sales
0,d_1,108
1,d_2,132
2,d_3,102
3,d_4,120
4,d_5,106


In [17]:
time_series.shape

(1941, 2)

In [18]:
time_series.tail()

,d,sales
1936,d_1937,69
1937,d_1938,75
1938,d_1939,110
1939,d_1940,156
1940,d_1941,99


## Merge Calendar Data

The day identifiers (`d_1`, `d_2`, ...) do not contain any temporal information by themselves. We merge the time series with the calendar dataset to obtain actual dates and calendar attributes that will be used for feature engineering.

In [19]:
time_series = time_series.merge(
    calendar,
    on="d",
    how="left"
)

time_series.shape

(1941, 15)

In [20]:
time_series.head()

,d,sales,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,d_1,108,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
1,d_2,132,2011-01-30,11101,Sunday,2,1,2011,NaN,NaN,NaN,NaN,0,0,0
2,d_3,102,2011-01-31,11101,Monday,3,1,2011,NaN,NaN,NaN,NaN,0,0,0
3,d_4,120,2011-02-01,11101,Tuesday,4,2,2011,NaN,NaN,NaN,NaN,1,1,0
4,d_5,106,2011-02-02,11101,Wednesday,5,2,2011,NaN,NaN,NaN,NaN,1,0,1


In [21]:
time_series = time_series.drop(
    columns=["snap_TX", "snap_WI"]
)

time_series.head()

,d,sales,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA
0,d_1,108,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0
1,d_2,132,2011-01-30,11101,Sunday,2,1,2011,NaN,NaN,NaN,NaN,0
2,d_3,102,2011-01-31,11101,Monday,3,1,2011,NaN,NaN,NaN,NaN,0
3,d_4,120,2011-02-01,11101,Tuesday,4,2,2011,NaN,NaN,NaN,NaN,1
4,d_5,106,2011-02-02,11101,Wednesday,5,2,2011,NaN,NaN,NaN,NaN,1


In [22]:
event_cols = [
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

time_series[event_cols] = (
    time_series[event_cols]
    .fillna("None")
)

time_series[event_cols].head()

,event_name_1,event_type_1,event_name_2,event_type_2
0,None,None,None,None
1,None,None,None,None
2,None,None,None,None
3,None,None,None,None
4,None,None,None,None


In [23]:
time_series["date"] = pd.to_datetime(time_series["date"])

In [24]:
time_series["quarter"] = time_series["date"].dt.quarter

time_series["is_weekend"] = (
    time_series["weekday"]
    .isin(["Saturday", "Sunday"])
    .astype(int)
)

In [25]:
time_series[
    ["date", "weekday", "month", "quarter", "is_weekend"]
].head()

,date,weekday,month,quarter,is_weekend
0,2011-01-29,Saturday,1,1,1
1,2011-01-30,Sunday,1,1,1
2,2011-01-31,Monday,1,1,0
3,2011-02-01,Tuesday,2,1,0
4,2011-02-02,Wednesday,2,1,0


## Create Lag Features

Past sales are often the strongest predictor of future demand. Lag features allow the model to use historical observations, enabling it to capture short-term trends, seasonality, and sales momentum.

In [26]:
time_series["lag_1"] = time_series["sales"].shift(1)

time_series["lag_7"] = time_series["sales"].shift(7)

time_series["lag_28"] = time_series["sales"].shift(28)

In [27]:
time_series[
    ["date","sales","lag_1","lag_7","lag_28"]
].head(35)

,date,sales,lag_1,lag_7,lag_28
0,2011-01-29,108,NaN,NaN,NaN
1,2011-01-30,132,108.0,NaN,NaN
2,2011-01-31,102,132.0,NaN,NaN
3,2011-02-01,120,102.0,NaN,NaN
4,2011-02-02,106,120.0,NaN,NaN
...,...,...,...,...,...
30,2011-02-28,0,0.0,0.0,102.0
31,2011-03-01,0,0.0,0.0,120.0
32,2011-03-02,0,0.0,0.0,106.0
33,2011-03-03,0,0.0,0.0,123.0


In [28]:
time_series.loc[20:35, ["date", "sales", "lag_1", "lag_7"]]

,date,sales,lag_1,lag_7
20,2011-02-18,0,0.0,0.0
21,2011-02-19,0,0.0,0.0
22,2011-02-20,0,0.0,0.0
23,2011-02-21,0,0.0,0.0
24,2011-02-22,0,0.0,0.0
...,...,...,...,...
31,2011-03-01,0,0.0,0.0
32,2011-03-02,0,0.0,0.0
33,2011-03-03,0,0.0,0.0
34,2011-03-04,0,0.0,0.0


## Create Rolling Features

Rolling statistics summarize recent sales behavior over a fixed time window. Unlike lag features, which capture individual past observations, rolling features describe short-term trends and help the model distinguish between temporary fluctuations and sustained demand patterns.

In [29]:
time_series["rolling_mean_7"] = (
    time_series["sales"]
    .shift(1)
    .rolling(window=7)
    .mean()
)

time_series["rolling_mean_28"] = (
    time_series["sales"]
    .shift(1)
    .rolling(window=28)
    .mean()
)

In [30]:
time_series[
    [
        "date",
        "sales",
        "rolling_mean_7",
        "rolling_mean_28"
    ]
].head(35)

,date,sales,rolling_mean_7,rolling_mean_28
0,2011-01-29,108,NaN,NaN
1,2011-01-30,132,NaN,NaN
2,2011-01-31,102,NaN,NaN
3,2011-02-01,120,NaN,NaN
4,2011-02-02,106,NaN,NaN
...,...,...,...,...
30,2011-02-28,0,0.0,43.250000
31,2011-03-01,0,0.0,39.607143
32,2011-03-02,0,0.0,35.321429
33,2011-03-03,0,0.0,31.535714


In [31]:
time_series["rolling_std_7"] = (
    time_series["sales"]
    .shift(1)
    .rolling(window=7)
    .std()
)

In [32]:
time_series[
    [
        "date",
        "sales",
        "rolling_mean_7",
        "rolling_std_7"
    ]
].head(35)

,date,sales,rolling_mean_7,rolling_std_7
0,2011-01-29,108,NaN,NaN
1,2011-01-30,132,NaN,NaN
2,2011-01-31,102,NaN,NaN
3,2011-02-01,120,NaN,NaN
4,2011-02-02,106,NaN,NaN
...,...,...,...,...
30,2011-02-28,0,0.0,0.0
31,2011-03-01,0,0.0,0.0
32,2011-03-02,0,0.0,0.0
33,2011-03-03,0,0.0,0.0


## Prepare Modeling Dataset

The feature engineering stage is now complete. Before training a forecasting model, we prepare the final dataset by handling missing values, selecting the required features, and separating the input features from the target variable.

In [33]:
time_series = time_series.dropna().reset_index(drop=True)

time_series.shape

(1913, 21)

In [34]:
time_series.isna().sum()

d                  0
sales              0
date               0
wm_yr_wk           0
weekday            0
                  ..
lag_7              0
lag_28             0
rolling_mean_7     0
rolling_mean_28    0
rolling_std_7      0
Length: 21, dtype: int64

## Encode Categorical Features

Some calendar variables are stored as text (categorical values), while most machine learning models require numerical inputs. We encode these categorical features into numerical representations without losing the underlying information.

In [35]:
categorical_cols = [
    "weekday",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

time_series = pd.get_dummies(
    time_series,
    columns=categorical_cols,
    drop_first=True
)

time_series.shape

(1913, 62)

## Define Final Feature Set

After encoding the categorical variables, we define the final feature matrix (`X`) and the target variable (`y`). At this stage, every feature is numerical and ready for model training.

In [36]:
feature_columns = time_series.columns.drop([
    "d",
    "date",
    "sales",
    "wm_yr_wk"
])

X = time_series[feature_columns]
y = time_series["sales"]

In [37]:
print("Number of features:", len(feature_columns))
print("X shape:", X.shape)
print("y shape:", y.shape)

X.head()

Number of features: 58
X shape: (1913, 58)
y shape: (1913,)


,wday,month,year,snap_CA,quarter,is_weekend,lag_1,lag_7,lag_28,rolling_mean_7,rolling_mean_28,rolling_std_7,weekday_Monday,weekday_Saturday,weekday_Sunday,weekday_Thursday,weekday_Tuesday,weekday_Wednesday,event_name_1_Christmas,event_name_1_Cinco De Mayo,event_name_1_ColumbusDay,event_name_1_Easter,event_name_1_Eid al-Fitr,event_name_1_EidAlAdha,event_name_1_Father's day,event_name_1_Halloween,event_name_1_IndependenceDay,event_name_1_LaborDay,event_name_1_LentStart,event_name_1_LentWeek2,event_name_1_MartinLutherKingDay,event_name_1_MemorialDay,event_name_1_Mother's day,event_name_1_NBAFinalsEnd,event_name_1_NBAFinalsStart,event_name_1_NewYear,event_name_1_None,event_name_1_OrthodoxChristmas,event_name_1_OrthodoxEaster,event_name_1_Pesach End,event_name_1_PresidentsDay,event_name_1_Purim End,event_name_1_Ramadan starts,event_name_1_StPatricksDay,event_name_1_SuperBowl,event_name_1_Thanksgiving,event_name_1_ValentinesDay,event_name_1_VeteransDay,event_type_1_National,event_type_1_None,event_type_1_Religious,event_type_1_Sporting,event_name_2_Easter,event_name_2_Father's day,event_name_2_None,event_name_2_OrthodoxEaster,event_type_2_None,event_type_2_Religious
0,1,2,2011,0,1,1,0.0,0.0,108.0,0.0,51.821429,0.0,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,True,False
1,2,2,2011,0,1,1,0.0,0.0,132.0,0.0,47.964286,0.0,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,True,False
2,3,2,2011,0,1,0,0.0,0.0,102.0,0.0,43.250000,0.0,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,True,False
3,4,3,2011,1,1,0,0.0,0.0,120.0,0.0,39.607143,0.0,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,True,False
4,5,3,2011,1,1,0,0.0,0.0,106.0,0.0,35.321429,0.0,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,True,False


In [38]:
time_series.to_csv("engineered_time_series.csv", index=False)

In [39]:
print("Final dataset shape:", time_series.shape)

time_series.head()

Final dataset shape: (1913, 62)


,d,sales,date,wm_yr_wk,wday,month,year,snap_CA,quarter,is_weekend,lag_1,lag_7,lag_28,rolling_mean_7,rolling_mean_28,rolling_std_7,weekday_Monday,weekday_Saturday,weekday_Sunday,weekday_Thursday,weekday_Tuesday,weekday_Wednesday,event_name_1_Christmas,event_name_1_Cinco De Mayo,event_name_1_ColumbusDay,event_name_1_Easter,event_name_1_Eid al-Fitr,event_name_1_EidAlAdha,event_name_1_Father's day,event_name_1_Halloween,event_name_1_IndependenceDay,event_name_1_LaborDay,event_name_1_LentStart,event_name_1_LentWeek2,event_name_1_MartinLutherKingDay,event_name_1_MemorialDay,event_name_1_Mother's day,event_name_1_NBAFinalsEnd,event_name_1_NBAFinalsStart,event_name_1_NewYear,event_name_1_None,event_name_1_OrthodoxChristmas,event_name_1_OrthodoxEaster,event_name_1_Pesach End,event_name_1_PresidentsDay,event_name_1_Purim End,event_name_1_Ramadan starts,event_name_1_StPatricksDay,event_name_1_SuperBowl,event_name_1_Thanksgiving,event_name_1_ValentinesDay,event_name_1_VeteransDay,event_type_1_National,event_type_1_None,event_type_1_Religious,event_type_1_Sporting,event_name_2_Easter,event_name_2_Father's day,event_name_2_None,event_name_2_OrthodoxEaster,event_type_2_None,event_type_2_Religious
0,d_29,0,2011-02-26,11105,1,2,2011,0,1,1,0.0,0.0,108.0,0.0,51.821429,0.0,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,True,False
1,d_30,0,2011-02-27,11105,2,2,2011,0,1,1,0.0,0.0,132.0,0.0,47.964286,0.0,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,True,False
2,d_31,0,2011-02-28,11105,3,2,2011,0,1,0,0.0,0.0,102.0,0.0,43.250000,0.0,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,True,False
3,d_32,0,2011-03-01,11105,4,3,2011,1,1,0,0.0,0.0,120.0,0.0,39.607143,0.0,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,True,False
4,d_33,0,2011-03-02,11105,5,3,2011,1,1,0,0.0,0.0,106.0,0.0,35.321429,0.0,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,True,False
